# Lossless byte-level BPE — Maick Dane Nkou

This notebook **retrains** the submission; it does not patch a decoder onto the old vocabulary.
The old `c41-lower-alph500-b3` recipe removed case and whitespace. Its reported
1.7440 was only the base fertility score, not the final score with reconstruction.

The replacement uses **no normalizer**, a matched ByteLevel pre-tokenizer/decoder,
all 256 byte-alphabet symbols, and **no special tokens**. It preserves case,
Unicode representation, punctuation, spaces, tabs and newlines. Training uses only
the official **train** split; validation is never passed to the trainer.

**Run all in Google Colab (CPU is sufficient).** Downloads and training require
network access. Export is blocked unless the reloaded model reconstructs every
validation row exactly and passes the pinned official checker. No new score is
claimed before this run completes. The existing repository tokenizer remains the
legacy artifact until you replace it with the checked export from this notebook.

The original recipe is available in Git at commit
`6a10f8b5db317e15ba14be6232440f72a4f6bae1` for provenance.

## 1. Dependencies
The exact tokenizers version is part of the submission contract.

In [ ]:
%pip install -q "tokenizers==0.22.1" "datasets>=4,<5" "PyYAML==6.0.2"
import tokenizers
assert tokenizers.__version__ == "0.22.1", "Restart the runtime after installing dependencies"
print("tokenizers:", tokenizers.__version__)

## 2. Reversible recipe and export gates
The byte alphabet is an encoding definition, not a pretrained vocabulary.
No lowercasing, NFC conversion, accent stripping, whitespace stripping, or
post-processor is applied. No special tokens are reserved: literal text such as
`[UNK]` must survive decoding even with `skip_special_tokens=True`.

The losslessness gate below is **stricter** than the pinned official metric:
it compares the original strings without NFC normalization or trimming.

In [ ]:
import hashlib
import importlib.util
import json
import math
import random
import shutil
import tempfile
import time
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
DATASET_ID = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"
RECIPE = "lossless-bytelevel-b3"
BOOST = {"ha": 3, "sw": 3, "yo": 3, "am": 3}
VOCAB_SIZE = 10_000
MIN_FREQUENCY = 5
MAX_FILE_BYTES = 20 * 1024 * 1024
TEAM_SLUG = "maick-dane-nkou"
# Generated data/models/reports stay outside the Git submission directory.
RUN_DIR = Path.cwd() / "artifacts" / RECIPE
RUN_DIR.mkdir(parents=True, exist_ok=True)


def new_tokenizer():
    # Every UTF-8 byte is representable, so an UNK token is unnecessary.
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)
    tok.decoder = decoders.ByteLevel()
    return tok


def train_tokenizer(texts, *, vocab_size=VOCAB_SIZE, min_frequency=MIN_FREQUENCY):
    tok = new_tokenizer()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()),
        special_tokens=[],
        show_progress=True,
    )
    tok.train_from_iterator(texts, trainer=trainer)
    assert tok.normalizer is None
    assert tok.get_vocab_size(with_added_tokens=True) <= vocab_size
    assert set(pre_tokenizers.ByteLevel.alphabet()) <= set(tok.get_vocab())
    return tok


def corpus_iterator(train_by_lang):
    """Stable round-robin; only the train split is passed here."""
    iterators = {lang: iter(train_by_lang[lang]) for lang in LANGUAGES}
    active = list(LANGUAGES)
    while active:
        for lang in active.copy():
            try:
                text = next(iterators[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(BOOST.get(lang, 1)):
                yield text  # original string, with all case/whitespace intact


def assert_exact_roundtrip(tok, texts, *, batch_size=512):
    """Fail on any lost character, including boundary whitespace or special text."""
    failures = 0
    first_indices = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tok.encode_batch(batch, add_special_tokens=False)
        ids = [enc.ids for enc in encoded]
        decoded = tok.decode_batch(ids, skip_special_tokens=False)
        skipped = tok.decode_batch(ids, skip_special_tokens=True)
        for i, (original, restored, restored_skip) in enumerate(zip(batch, decoded, skipped, strict=True)):
            if original != restored or original != restored_skip:
                failures += 1
                if len(first_indices) < 5:
                    first_indices.append(start + i)
    if failures:
        raise ValueError(f"Export blocked: {failures}/{len(texts)} lossy rows; first indices {first_indices}")
    return len(texts)


def assert_submission_ready(report, expected_rows):
    # The official `valid` flag alone does NOT reject lossy models.
    if not report.get("valid"):
        raise ValueError(f"Official validity checks failed: {report.get('errors')}")
    if report.get("rows") != expected_rows:
        raise ValueError("Official checker did not evaluate the complete validation split")
    if (report.get("lossy_rows") != 0 or report.get("reconstruction") != 1.0
            or report.get("reconstruction_penalty") != 0.0):
        raise ValueError("Export blocked: official reconstruction is not 100%")
    if set(report.get("fertility", {})) != set(LANGUAGES):
        raise ValueError("Missing validation language")
    if set(report.get("unknown_rate", {})) != set(LANGUAGES):
        raise ValueError("Missing unknown-token measurements")
    if any(rate != 0.0 for rate in report["unknown_rate"].values()):
        raise ValueError("Export blocked: unknown tokens were emitted")
    for key in ("score", "guardrail_penalty", "reconstruction_penalty"):
        if not math.isfinite(report[key]) or report[key] < 0:
            raise ValueError(f"Invalid metric: {key}")
    # Guardrail overages are allowed by the competition but must be included.
    base = sum(report["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
    expected = base + report["guardrail_penalty"] + report["reconstruction_penalty"]
    if not math.isclose(report["score"], expected, rel_tol=1e-12, abs_tol=1e-12):
        raise ValueError("Score is missing a penalty")

## 3. Regression tests (not competition training)
These small, disposable models test the implementation only. Their vocabulary
and merge table are **never reused** by the final training run. Tests cover all
six languages, case, NFC/NFD, repeated spaces, tabs, line breaks, emoji, literal
special-token strings, and previously unseen Unicode characters. They also
verify that lowercasing, a mismatched decoder, and artificial prefix spaces are rejected.

In [ ]:
SMOKE_TEXTS = [
    "Knowledge grows when it is shared.",
    "Le savoir grandit lorsqu’il est partagé.",
    "Ilimi yana ƙaruwa idan an raba shi.",
    "Maarifa hukua yanaposhirikishwa.",
    "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.",
    "እውቀት ሲካፈል ያድጋል።",
]
ROUNDTRIP_CASES = SMOKE_TEXTS + [
    "", " ", "   ", "\t\n\r\n", "  Hello  WORLD!\tNext\nline.  ",
    "[UNK] [CLS] [SEP] <0xFF> <s>", "é e\u0301 Ì I\u0300",
    "👩🏿‍💻 🌍 中文 العربية", "a\u00a0b\u2003c\u200bd",
    "\x00\x01\x7f\ufeff\U0010ffff", "don't l’amour — … ።",
]
ROUNDTRIP_CASES += [unicodedata.normalize("NFD", text) for text in SMOKE_TEXTS]
rng = random.Random(41)
scalars = []
while len(scalars) < 256:
    point = rng.randrange(0x110000)
    if not 0xD800 <= point <= 0xDFFF:  # Python surrogates are not Unicode scalar values
        scalars.append(chr(point))
ROUNDTRIP_CASES += ["".join(scalars), " ".join(scalars)]


def expect_rejected(action):
    try:
        action()
    except ValueError:
        return
    raise AssertionError("Regression: a lossy/invalid candidate was accepted")


def run_regression_tests():
    from tokenizers import normalizers
    mini = train_tokenizer(iter(SMOKE_TEXTS), vocab_size=512, min_frequency=1)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "tokenizer.json"
        mini.save(str(path))
        loaded = Tokenizer.from_file(str(path))
        assert_exact_roundtrip(loaded, ROUNDTRIP_CASES)
        for mutation in ("lowercase", "wrong_decoder", "prefix_space", "whitespace_split"):
            broken = Tokenizer.from_str(loaded.to_str())
            if mutation == "lowercase":
                broken.normalizer = normalizers.Lowercase()
            elif mutation == "wrong_decoder":
                broken.decoder = decoders.ByteFallback()
            elif mutation == "prefix_space":
                broken.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
            else:
                broken.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
            expect_rejected(lambda: assert_exact_roundtrip(broken, ROUNDTRIP_CASES))
    good = {
        "valid": True, "rows": 24_000, "lossy_rows": 0, "reconstruction": 1.0,
        "reconstruction_penalty": 0.0, "guardrail_penalty": 0.2, "score": 2.2,
        "fertility": dict.fromkeys(LANGUAGES, 2.0),
        "unknown_rate": dict.fromkeys(LANGUAGES, 0.0),
        "penalised": dict.fromkeys(LANGUAGES, 2.0),
    }
    assert_submission_ready(good, 24_000)
    for patch in (
        {"lossy_rows": 1}, {"reconstruction": 0.99}, {"reconstruction_penalty": 0.1},
        {"valid": False}, {"rows": 6}, {"score": 2.0}, {"score": float("nan")},
        {"unknown_rate": dict.fromkeys(LANGUAGES, 0.1)},
    ):
        expect_rejected(lambda: assert_submission_ready({**good, **patch}, 24_000))
    print(f"Regression tests passed ({len(ROUNDTRIP_CASES)} exact round-trip cases).")

run_regression_tests()

## 4. Pinned official checker — no silent fallback
The old notebook could use a stale `utils.py` and its copied metric omitted penalties.
Here the checker is downloaded from a fixed official Git commit, its SHA-256 is
verified, and that exact module is loaded directly. A failed download stops the run;
an old local `utils.py` is never imported. The pin documents the scoring version;
organizers may update their rules later.

In [ ]:
OFFICIAL_COMMIT = "75578f2400c39b1f8e31ce7e7104b37fbc470d11"
OFFICIAL_UTILS_SHA256 = "1727de34136097eb48addabf90501589bdfefa31c20e201bab53c38f2f7c9688"
OFFICIAL_UTILS_URL = (
    "https://raw.githubusercontent.com/aims-ai-research-foundations/"
    f"airf-multilingual-tokenizer-challenge/{OFFICIAL_COMMIT}/starter/utils.py"
)
with urllib.request.urlopen(OFFICIAL_UTILS_URL, timeout=60) as response:
    helper = response.read()
if hashlib.sha256(helper).hexdigest() != OFFICIAL_UTILS_SHA256:
    raise RuntimeError("Official checker hash mismatch; do not continue")
helper_path = RUN_DIR / "official_utils.py"
helper_path.write_bytes(helper)
spec = importlib.util.spec_from_file_location("pinned_official_utils", helper_path)
official_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official_utils)
assert official_utils.REQUIRED_TOKENIZERS_VERSION == "0.22.1"
assert official_utils.RECONSTRUCTION_PENALTY == 3.0
print("Official checker:", OFFICIAL_COMMIT)

## 5. Load only the official public train and validation splits
No external corpus, pretrained tokenizer, or vocabulary is used. The release
must contain 40,000 training and 4,000 validation rows per language. Input text
is never stripped or normalized. There is no reduced-data mode for final export.

In [ ]:
from datasets import load_dataset

train_data, validation_data = load_dataset(
    DATASET_ID, revision=DATASET_REVISION,
    split=["train", "validation"], cache_dir=str(RUN_DIR / "dataset-cache"),
)
train_by_lang = {lang: [] for lang in LANGUAGES}
for row in train_data:
    if row["language"] not in train_by_lang or not isinstance(row["text"], str):
        raise ValueError("Unexpected training row")
    train_by_lang[row["language"]].append(row["text"])
val_rows = [(row["language"], row["text"]) for row in validation_data]
assert {lang: len(texts) for lang, texts in train_by_lang.items()} == dict.fromkeys(LANGUAGES, 40_000)
assert Counter(lang for lang, _ in val_rows) == dict.fromkeys(LANGUAGES, 4_000)
assert all(isinstance(text, str) and text.split() for _, text in val_rows)
validation_texts = [text for _, text in val_rows]
print("Train:", len(train_data), "| Validation:", len(val_rows))

## 6. Train from scratch and reload the candidate
This tokenizer is a fresh instance. Only `train_by_lang` is used to train it;
regression-test text and validation text do not enter training.

In [ ]:
started = time.perf_counter()
tokenizer = train_tokenizer(corpus_iterator(train_by_lang))
training_seconds = time.perf_counter() - started
assert tokenizer.get_vocab_size(with_added_tokens=True) == VOCAB_SIZE
candidate_dir = RUN_DIR / "candidate"
candidate_dir.mkdir(parents=True, exist_ok=True)
candidate_path = candidate_dir / "tokenizer.json"
tokenizer.save(str(candidate_path))
assert candidate_path.stat().st_size <= MAX_FILE_BYTES
# Verify the actual serialized artifact, not only the in-memory trainer result.
tokenizer = Tokenizer.from_file(str(candidate_path))
assert_exact_roundtrip(tokenizer, ROUNDTRIP_CASES)
checked_rows = assert_exact_roundtrip(tokenizer, validation_texts)
print(f"Trained in {training_seconds:.1f}s; {checked_rows:,} validation rows reconstructed exactly.")

## 7. Full official score, including BOTH penalties
The official report is the source of the score. A nonzero guardrail penalty is
reported and included (it does not invalidate the file). Any reconstruction
loss blocks export, even when the checker prints `READY FOR SUBMISSION`.

In [ ]:
official_report = official_utils.profile_submission(candidate_path, data=val_rows, repeats=3)
assert_submission_ready(official_report, len(val_rows))
base_score = sum(official_report["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
print(f"Base score:              {base_score:.6f}")
print(f"Guardrail penalty:       {official_report['guardrail_penalty']:.6f}")
print(f"Reconstruction penalty:  {official_report['reconstruction_penalty']:.6f}")
print(f"Full validation score:   {official_report['score']:.6f}")
print("Strict exact reconstruction: 100% (all original characters preserved)")

report = {
    "recipe": RECIPE,
    "training": {"vocab_size": VOCAB_SIZE, "min_frequency": MIN_FREQUENCY,
                 "boost": BOOST, "seconds": training_seconds, "normalizer": None,
                 "pre_tokenizer": "ByteLevel(add_prefix_space=False, use_regex=True)",
                 "decoder": "ByteLevel", "special_tokens": []},
    "dataset": {"id": DATASET_ID, "revision": DATASET_REVISION,
                "train_rows": len(train_data), "validation_rows": len(val_rows),
                "train_fingerprint": train_data._fingerprint,
                "validation_fingerprint": validation_data._fingerprint},
    "official_checker_commit": OFFICIAL_COMMIT,
    "official_checker_sha256": OFFICIAL_UTILS_SHA256,
    "tokenizer_sha256": hashlib.sha256(candidate_path.read_bytes()).hexdigest(),
    "tokenizers_version": tokenizers.__version__,
    "strict_lossy_rows": 0,
    "official": official_report,
}
report_path = RUN_DIR / "validation_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Measured report:", report_path)

## 8. Export only the verified candidate
Recheck the file hash, round-trip, and official gate immediately before copying.
The export does not overwrite the existing repository submission. Replace its
`tokenizer.json`, `metadata.yml` and `README.md` with these generated files **only
after a successful full run**, and include this notebook as `notebook.ipynb`.
Do not commit dataset caches, reports, or helper modules into the team directory.

In [ ]:
import yaml

assert_submission_ready(official_report, 24_000)
if hashlib.sha256(candidate_path.read_bytes()).hexdigest() != report["tokenizer_sha256"]:
    raise ValueError("Candidate changed since evaluation; rerun the evaluation")
assert_exact_roundtrip(Tokenizer.from_file(str(candidate_path)), validation_texts)
export_dir = RUN_DIR / "export" / TEAM_SLUG
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(candidate_path, export_dir / "tokenizer.json")
metadata = {
    "team": "Maick Dane Nkou", "members": ["Maick Dane Nkou"],
    "affiliation": "AIMS SOUTH AFRICA",
    "approach": (f"Lossless byte-level BPE 10k, no normalization, full byte alphabet; "
                 f"official train only, ha/sw/yo/am x3; validation score "
                 f"{official_report['score']:.4f}, exact reconstruction 100%."),
}
(export_dir / "metadata.yml").write_text(yaml.safe_dump(metadata, sort_keys=False), encoding="utf-8")
readme = [
    "# Maick Dane Nkou — lossless byte-level BPE", "",
    "Trained from scratch on the official train split only; no pretrained tokenizer or external corpus.",
    "", "## Recipe", "",
    "- BPE, 10,000 vocabulary entries, min_frequency=5, tokenizers==0.22.1",
    "- No normalizer, special tokens, or post-processor",
    "- ByteLevel(add_prefix_space=False, use_regex=True) + ByteLevel decoder",
    "- Complete 256-symbol byte alphabet; scored languages oversampled x3",
    f"- Dataset: `{DATASET_ID}` @ `{DATASET_REVISION}`",
    "", "## Measured validation results (24,000 rows; not hidden-test scores)", "",
    "| Language | Tokens/word | UNK rate |", "| --- | ---: | ---: |",
]
for lang in LANGUAGES:
    readme.append(f"| {lang} | {official_report['fertility'][lang]:.6f} | {official_report['unknown_rate'][lang]:.6f} |")
readme += [
    "", f"- Base score: {base_score:.6f}",
    f"- Guardrail penalty: {official_report['guardrail_penalty']:.6f}",
    f"- Reconstruction penalty: {official_report['reconstruction_penalty']:.6f}",
    f"- **Full validation score: {official_report['score']:.6f}**",
    "- Strict exact reconstruction: 24,000 / 24,000 rows (100%)",
    f"- Official checker commit: `{OFFICIAL_COMMIT}`",
    f"- Tokenizer SHA-256: `{report['tokenizer_sha256']}`",
    "", "## Reproduce", "",
    "Run notebook.ipynb end-to-end in Colab. It trains only on train, verifies the reloaded",
    "artifact on validation, runs the pinned official checker, and blocks lossy exports.",
    "BPE merge ties can vary across training runs; evaluate every regenerated artifact.", "",
]
(export_dir / "README.md").write_text("\n".join(readme), encoding="utf-8")
print("VERIFIED export:", export_dir)
print("Copy its three files into submissions/maick-dane-nkou/ and include this notebook.")
try:
    from google.colab import files
except ImportError:
    print("Not in Colab: retrieve the files from the export directory.")
else:
    for name in ("tokenizer.json", "metadata.yml", "README.md"):
        files.download(str(export_dir / name))